In [2]:
# ============================================================
# STEP 1: MOUNT GOOGLE DRIVE
# ============================================================
# This notebook reads the cleaned NHS articles and converts
# them into smaller chunks suitable for vector embeddings.
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# STEP 2: INSTALL REQUIRED LIBRARY
# ============================================================
# LangChain provides high-quality text splitters that preserve
# semantic structure better than simple string splitting.
# ============================================================

!pip install -q langchain-text-splitters

In [4]:
# ============================================================
# STEP 3: IMPORT REQUIRED LIBRARIES
# ============================================================

from pathlib import Path
from tqdm import tqdm
import json

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
# ============================================================
# STEP 4: DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Healthcare_RAG")

CLEANED_DIR = PROJECT_DIR / "data" / "cleaned"
CHUNK_DIR = PROJECT_DIR / "data" / "chunks"

# Create output folder if it doesn't exist
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

print("Cleaned Folder :", CLEANED_DIR)
print("Chunk Folder   :", CHUNK_DIR)

Cleaned Folder : /content/drive/MyDrive/Healthcare_RAG/data/cleaned
Chunk Folder   : /content/drive/MyDrive/Healthcare_RAG/data/chunks


In [6]:
# ============================================================
# STEP 5: LOAD CLEANED ARTICLES
# ============================================================
# This cell loads every cleaned NHS article available for
# chunking.
# ============================================================

cleaned_files = sorted(CLEANED_DIR.glob("*.txt"))

print("=" * 60)
print("CLEANED DATASET SUMMARY")
print("=" * 60)

print(f"Articles Found : {len(cleaned_files)}")

print("\nFirst 10 Articles:")

for file in cleaned_files[:10]:
    print(file.name)

print("=" * 60)

CLEANED DATASET SUMMARY
Articles Found : 698

First 10 Articles:
aciclovir_zovirax.txt
acrivastine-brand_name_benadryl_allergy_relief.txt
acute_lymphoblastic_leukaemia.txt
adalimumab.txt
alendronic_acid-brand_names_fosamax_binosto.txt
allopurinol-zyloric_uricto.txt
amlodipine.txt
amoxicillin.txt
anal_pain.txt
anastrozole-brand_name_arimidex.txt


In [7]:
# ============================================================
# STEP 6: CREATE THE TEXT SPLITTER
# ============================================================
# This splitter divides long NHS articles into smaller chunks.
#
# Parameters:
#   chunk_size     : Maximum characters per chunk.
#   chunk_overlap  : Overlapping characters between chunks.
#
# The overlap helps preserve context across chunk boundaries,
# improving retrieval quality in the RAG pipeline.
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(

    # Maximum size of each chunk
    chunk_size=1000,

    # Number of overlapping characters
    chunk_overlap=200,

    # Priority order for splitting
    separators=[
        "\n## ",   # Section headings
        "\n### ",  # Subsection headings
        "\n\n",    # Paragraphs
        "\n",      # Lines
        ". ",      # Sentences
        " ",       # Words
        ""         # Character-level fallback
    ]

)

print("=" * 60)
print("TEXT SPLITTER CREATED SUCCESSFULLY")
print("=" * 60)
print(f"Chunk Size    : {text_splitter._chunk_size}")
print(f"Chunk Overlap : {text_splitter._chunk_overlap}")
print("=" * 60)

TEXT SPLITTER CREATED SUCCESSFULLY
Chunk Size    : 1000
Chunk Overlap : 200


In [8]:
# ============================================================
# STEP 7: GENERATE CHUNKS FROM ALL ARTICLES
# ============================================================
# This cell:
#   1. Reads every cleaned NHS article.
#   2. Splits the article into chunks.
#   3. Creates metadata for each chunk.
#   4. Saves each article's chunks as a JSON file.
#   5. Displays processing statistics.
#
# Output:
#   data/chunks/
#       asthma.json
#       diabetes.json
#       migraine.json
#       ...
# ============================================================

# Statistics
total_articles = 0
total_chunks = 0

print("=" * 70)
print("STARTING CHUNK GENERATION")
print("=" * 70)

# Process every cleaned article
for file_path in tqdm(cleaned_files):

    # Read article
    with open(file_path, "r", encoding="utf-8") as file:
        article = file.read()

    # Split article into chunks
    chunks = text_splitter.split_text(article)

    # Store chunk data
    article_chunks = []

    # Create metadata for every chunk
    for index, chunk in enumerate(chunks, start=1):

        article_chunks.append({

            "chunk_id": index,

            "source_file": file_path.name,

            "chunk_length": len(chunk),

            "text": chunk

        })

    # Save JSON file
    output_file = CHUNK_DIR / f"{file_path.stem}.json"

    with open(output_file, "w", encoding="utf-8") as file:

        json.dump(
            article_chunks,
            file,
            indent=4,
            ensure_ascii=False
        )

    total_articles += 1
    total_chunks += len(article_chunks)

# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("CHUNK GENERATION COMPLETED")
print("=" * 70)

print(f"Articles Processed : {total_articles}")
print(f"Total Chunks       : {total_chunks}")
print(f"Average Chunks     : {total_chunks / total_articles:.2f}")

print("=" * 70)

STARTING CHUNK GENERATION


100%|██████████| 698/698 [03:26<00:00,  3.39it/s]



CHUNK GENERATION COMPLETED
Articles Processed : 698
Total Chunks       : 3796
Average Chunks     : 5.44


In [9]:
# ============================================================
# STEP 8: VALIDATE GENERATED CHUNKS
# ============================================================
# This cell:
#   1. Opens one generated JSON file.
#   2. Displays chunk statistics.
#   3. Shows the first few chunks.
#   4. Confirms that chunking worked correctly.
# ============================================================

# Select the first JSON chunk file
sample_chunk_file = sorted(CHUNK_DIR.glob("*.json"))[0]

print("=" * 70)
print("VALIDATING GENERATED CHUNKS")
print("=" * 70)

print(f"Sample File : {sample_chunk_file.name}")

# Load JSON
with open(sample_chunk_file, "r", encoding="utf-8") as file:
    chunk_data = json.load(file)

print(f"Total Chunks : {len(chunk_data)}")

print("\n")

# Display the first three chunks
for chunk in chunk_data[:3]:

    print("=" * 70)
    print(f"Chunk ID      : {chunk['chunk_id']}")
    print(f"Source File   : {chunk['source_file']}")
    print(f"Chunk Length  : {chunk['chunk_length']} characters")

    print("\nChunk Preview:\n")

    print(chunk["text"][:500])

    print("\n")

VALIDATING GENERATED CHUNKS
Sample File : aciclovir_zovirax.json
Total Chunks : 14


Chunk ID      : 1
Source File   : aciclovir_zovirax.txt
Chunk Length  : 456 characters

Chunk Preview:

# Aciclovir (Zovirax)

Aciclovir is an antiviral medicine used to treat certain infections. Most types are only available with a prescription, but you can buy some types from shops and pharmacies.

## What aciclovir is for

Aciclovir is used to treat infections such as:

• cold sores

• shingles

• chickenpox

• genital herpes

• herpes simplex eye infections

It's also sometimes used to prevent these infections in people with a weakened immune system.


Chunk ID      : 2
Source File   : aciclovir_zovirax.txt
Chunk Length  : 515 characters

Chunk Preview:

## How to use aciclovir

There are several different types of aciclovir, including:

• tablets

• cream

• eye ointment

How you use your medicine and how much to use depends on which type it is and how much aciclovir it contains.

Always check the

In [10]:
# ============================================================
# STEP 9: CHUNK DATASET STATISTICS
# ============================================================
# This cell calculates statistics for the entire chunk dataset.
#
# Statistics:
#   - Total chunk files
#   - Total chunks
#   - Average chunk size
#   - Minimum chunk size
#   - Maximum chunk size
# ============================================================

total_chunk_files = 0
total_chunks = 0

chunk_lengths = []

for json_file in CHUNK_DIR.glob("*.json"):

    with open(json_file, "r", encoding="utf-8") as file:
        chunks = json.load(file)

    total_chunk_files += 1

    for chunk in chunks:

        total_chunks += 1
        chunk_lengths.append(chunk["chunk_length"])

print("=" * 70)
print("CHUNK DATASET STATISTICS")
print("=" * 70)

print(f"Articles                : {total_chunk_files}")
print(f"Total Chunks            : {total_chunks}")
print(f"Average Chunk Length    : {sum(chunk_lengths)/len(chunk_lengths):.2f}")
print(f"Smallest Chunk          : {min(chunk_lengths)}")
print(f"Largest Chunk           : {max(chunk_lengths)}")

print("=" * 70)

CHUNK DATASET STATISTICS
Articles                : 698
Total Chunks            : 3796
Average Chunk Length    : 645.59
Smallest Chunk          : 16
Largest Chunk           : 999
